In [ ]:
# Install required packages
!pip install -q -U transformers datasets sacrebleu sentencepiece accelerate bitsandbytes
print("✅ Packages installed")

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

In [ ]:
import torch

print(f"🖥️  CUDA Available : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"🚀 GPU             : {torch.cuda.get_device_name(0)}")
    print(f"💾 GPU Memory      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU found — enable it in Settings > Accelerator > GPU")

In [ ]:
# ============================================================
# ✅ Kaggle + NLLB Configuration
# ============================================================

# --- NLLB Language codes ---
SRC_LANG = "eng_Latn"
TGT_LANG = "mar_Deva"

# --- Column names (UPDATE if needed) ---
SRC_COL = "english_legal_data"
TGT_COL = "benchmark_translation"

# --- Kaggle dataset path (from your screenshot) ---
KAGGLE_INPUT = "/kaggle/input/datasets/nujipndt51/legal-nllb-dataset"

TRAIN_CSV = f"{KAGGLE_INPUT}/legal_train.csv"
TEST_CSV  = f"{KAGGLE_INPUT}/legal_test.csv"
VAL_CSV   = f"{KAGGLE_INPUT}/legal_val.csv"

# --- Model ---
MODEL_NAME = "facebook/nllb-200-distilled-600M"

# --- Output path (Kaggle writable) ---
OUTPUT_DIR = "/kaggle/working/nllb_model"

# --- Training params ---
MAX_INPUT_LEN   = 128
MAX_TARGET_LEN  = 128
BATCH_SIZE      = 8
GRAD_ACCUM      = 4
NUM_EPOCHS      = 3
LEARNING_RATE   = 5e-5
BLEU_SAMPLE_SIZE = 500  
# ============================================================
print("✅ CONFIG READY")
print("Train:", TRAIN_CSV)
print("Test :", TEST_CSV)
print("Val  :", VAL_CSV)
print("Model:", MODEL_NAME)

In [ ]:
import os
import pandas as pd

# 🔍 Recursively search for the exact folder containing your files
base_path = ""
for root, dirs, files in os.walk("/kaggle/input"):
    if "legal_train.csv" in files:
        base_path = root
        break

if not base_path:
    raise FileNotFoundError("Could not find 'legal_train.csv' anywhere inside /kaggle/input/. Are you absolutely sure the dataset is attached to this notebook?")

print(f"✅ Found exact dataset path: {base_path}")

# ✅ Auto-assigned paths
TRAIN_CSV = f"{base_path}/legal_train.csv"
TEST_CSV  = f"{base_path}/legal_test.csv"
VAL_CSV   = f"{base_path}/legal_val.csv"

# ✅ Correct column names
SRC_COL = "english_legal_data"
TGT_COL = "benchmark_translation"

def load_csv(path, src_col, tgt_col):
    df = pd.read_csv(path)

    print(f"\n📂 Loaded  : {path}")
    print(f"📊 Shape   : {df.shape}")
    print(f"📑 Columns : {list(df.columns)}")

    # Validate
    assert src_col in df.columns, f"❌ '{src_col}' not found. Available columns: {list(df.columns)}"
    assert tgt_col in df.columns, f"❌ '{tgt_col}' not found. Available columns: {list(df.columns)}"

    # Clean
    df = df[[src_col, tgt_col]].copy()
    df = df.dropna(subset=[src_col, tgt_col])

    df[src_col] = df[src_col].astype(str).str.strip()
    df[tgt_col] = df[tgt_col].astype(str).str.strip()

    df = df[(df[src_col] != '') & (df[tgt_col] != '')]

    print(f"✅ Clean rows: {len(df)}")
    return df


print("🚀 Loading datasets...")

train_df = load_csv(TRAIN_CSV, SRC_COL, TGT_COL)
test_df  = load_csv(TEST_CSV,  SRC_COL, TGT_COL)
val_df   = load_csv(VAL_CSV,   SRC_COL, TGT_COL)

print(f"\n📦 Final Sizes → Train: {len(train_df)} | Test: {len(test_df)} | Val: {len(val_df)}")

print("\n📝 Sample data:")
print(train_df.head(3))


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️ Device: {device}")

BASE_MODEL = "facebook/nllb-200-distilled-600M"
SRC_LANG = "eng_Latn"
TGT_LANG = "mar_Deva"

print(f"⚙️ Loading {BASE_MODEL}...")

# NLLB requires the source language in the tokenizer initialization
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, src_lang=SRC_LANG)

# Load model in Float16 to save Kaggle GPU VRAM!
model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL).to(device)

print("✅ NLLB Model and Tokenizer loaded successfully!")


In [ ]:
import sacrebleu
from tqdm import tqdm

def evaluate_bleu(model, tokenizer, df, src_col, tgt_col, tgt_lang, max_length=128, batch_size=8):
    model.eval()
    # Sample 500 rows for speed
    sample_df = df.sample(n=min(500, len(df)), random_state=42).reset_index(drop=True)
    
    all_preds, all_refs = [], []
    
    # Tell NLLB clearly what language to output
    tgt_lang_id = tokenizer.convert_tokens_to_ids(tgt_lang)
    
    for i in tqdm(range(0, len(sample_df), batch_size), desc="Evaluating Base Model"):
        batch = sample_df.iloc[i : i + batch_size]
        src_texts, tgt_texts = batch[src_col].tolist(), batch[tgt_col].tolist()
        
        inputs = tokenizer(src_texts, return_tensors="pt", padding=True, truncation=True, max_length=max_length).to(device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                forced_bos_token_id=tgt_lang_id,
                max_length=max_length,
                num_beams=4,
            )
        
        preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        all_preds.extend(preds)
        all_refs.extend(tgt_texts)
        
    bleu = sacrebleu.corpus_bleu(all_preds, [all_refs])
    return bleu.score

base_bleu = evaluate_bleu(model, tokenizer, test_df, SRC_COL, TGT_COL, TGT_LANG)
print(f"\n🎯 Baseline BLEU Score (Before Training): {base_bleu:.2f}")


In [ ]:
from datasets import Dataset

MAX_INPUT_LEN = 128
MAX_TARGET_LEN = 128

def tokenize_batch(examples):
    # Pass target texts directly into text_target
    inputs = tokenizer(
        examples[SRC_COL],
        text_target=examples[TGT_COL],
        max_length=MAX_INPUT_LEN,
        max_target_length=MAX_TARGET_LEN,
        padding="max_length",
        truncation=True,
    )
    
    # Replace padding token ids in labels with -100 so the model ignores loss on padding
    inputs["labels"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in inputs["labels"]
    ]
    return inputs

print("🔄 Converting DataFrames to HuggingFace Datasets...")
train_dataset = Dataset.from_pandas(train_df[[SRC_COL, TGT_COL]].reset_index(drop=True))
val_dataset   = Dataset.from_pandas(val_df[[SRC_COL, TGT_COL]].reset_index(drop=True))

print("⚙️ Tokenizing datasets (takes 1-3 minutes)...")
tokenized_train = train_dataset.map(tokenize_batch, batched=True, batch_size=64, remove_columns=train_dataset.column_names)
tokenized_val   = val_dataset.map(tokenize_batch, batched=True, batch_size=64, remove_columns=val_dataset.column_names)

# Convert to PyTorch tensors
tokenized_train.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
tokenized_val.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

print("✅ Tokenization Complete!")


In [ ]:
import os
import shutil
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

# Save to Kaggle's working directory!
CHECKPOINT_DIR = "/kaggle/working/nllb-checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, label_pad_token_id=-100)

training_args = Seq2SeqTrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=1,                  
    
    # 📉 MEMORY OPTIMIZATION BLOCK 📉
    per_device_train_batch_size=4,       # Halved to massively drop VRAM limit
    per_device_eval_batch_size=4,        
    gradient_accumulation_steps=8,       # 4 x 8 = 32 effective batch size (maintains train quality)
    optim="adamw_8bit",                  # Turns on 8-bit memory optimizer
    fp16=True,                           # Mixed precision compression (the correct way to use float16!)
    gradient_checkpointing=True,         # Deletes intermediate memory and recomputes on the fly
    
    learning_rate=5e-5,                  
    warmup_steps=200,
    weight_decay=0.01,
    eval_strategy="steps",               
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,                  
    load_best_model_at_end=True,
    predict_with_generate=True,
    generation_max_length=128,
    report_to="none"                     
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,          
    data_collator=data_collator,
)

print(f"🏋️ Starting ultra-low-memory fine-tuning for NLLB on {len(tokenized_train)} legal examples...")
trainer.train()

# -----------------------------------------------------
# 1. Save final model directly
# -----------------------------------------------------
FINAL_MODEL_DIR = "/kaggle/working/nllb-legal-final"
trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)
print(f"\n📂 Model successfully saved to: {FINAL_MODEL_DIR}")

# -----------------------------------------------------
# 2. Hard ZIP compression so you never lose it again!
# -----------------------------------------------------
print("🗜️ Zipping all model files together...")
shutil.make_archive(
    base_name="/kaggle/working/nllb-legal-final", 
    format="zip", 
    root_dir=FINAL_MODEL_DIR
)
print("✅ SUCCESS! 'nllb-legal-final.zip' is firmly saved. You can download it directly from the Kaggle 'Output' side-panel!")


In [ ]:
import sacrebleu
import torch
from tqdm import tqdm

print("📥 Using the fully trained model securely out of your GPU memory...")

# 1. 🚀 Use the physical model that survived training
ft_model = model
ft_tokenizer = tokenizer

# Cast to float16 for hyper-fast evaluating
ft_model.half()
ft_model.eval()

# 2. ⚙️ Define the exact evaluate function
def evaluate_bleu_full(model, tokenizer, df, src_col, tgt_col, tgt_lang, sample_size=500, max_length=128, batch_size=8):
    sample_df = df.sample(n=min(sample_size, len(df)), random_state=42).reset_index(drop=True)
    all_preds, all_refs = [], []
    
    tgt_lang_id = tokenizer.convert_tokens_to_ids(tgt_lang)
    
    for i in tqdm(range(0, len(sample_df), batch_size), desc="Evaluating Fine-Tuned Model"):
        batch = sample_df.iloc[i : i + batch_size]
        src_texts, tgt_texts = batch[src_col].tolist(), batch[tgt_col].tolist()
        
        inputs = tokenizer(src_texts, return_tensors="pt", padding=True, truncation=True, max_length=max_length).to("cuda")
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                forced_bos_token_id=tgt_lang_id,
                max_length=max_length,
                num_beams=4,
            )
        
        preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        all_preds.extend(preds)
        all_refs.extend(tgt_texts)
        
    bleu = sacrebleu.corpus_bleu(all_preds, [all_refs])
    return bleu.score, all_preds, all_refs, sample_df

BLEU_SAMPLE_SIZE = 500

print(f"\n📊 Evaluating FINE-TUNED NLLB on {BLEU_SAMPLE_SIZE} test samples...")
print("-" * 50)

# 3. 🚀 Process the Evaluation
ft_bleu, ft_preds, ft_refs, ft_sample_df = evaluate_bleu_full(
    ft_model, ft_tokenizer, test_df,
    SRC_COL, TGT_COL, "mar_Deva",
    sample_size=BLEU_SAMPLE_SIZE
)

# 4. 📈 Results Output Summary
print("\n" + "="*55)
print("       📈  BLEU SCORE COMPARISON")
print("="*55)
print("  Language Pair   : English (eng_Latn) → Marathi (mar_Deva)")
print(f"  Eval samples    : {BLEU_SAMPLE_SIZE}")
print("-" * 55)

try:
    print(f"  Base Model BLEU : {base_bleu:.2f}")
    improvement = ft_bleu - base_bleu
    emoji = "🚀" if improvement > 0 else "⚠️"
    print(f"  Fine-tuned BLEU : {ft_bleu:.2f} (Tested safely from memory)")
    print(f"  Improvement     : {emoji} {improvement:+.2f} BLEU points")
except NameError:
    print("  Base Model BLEU : [Not calculated before training]")
    print(f"  Fine-tuned BLEU : {ft_bleu:.2f} 🚀 (Tested safely from memory)")

print("="*55)

print("\n📝 Sample Translations (Fine-tuned Model):")
for i in range(min(5, len(ft_preds))):
    print(f"  [{i+1}] Source    : {ft_sample_df.iloc[i][SRC_COL][:80]}")
    print(f"       Reference : {ft_refs[i][:80]}")
    print(f"       Predicted : {ft_preds[i][:80]}")
    print()


In [ ]:
import os
print(os.listdir("/kaggle/working/nllb-legal-final"))


In [ ]:
import shutil

print("🗜️ Zipping the model files...")
# This creates a file named 'nllb-legal-final.zip' in your working directory
shutil.make_archive(
    base_name="/kaggle/working/nllb-legal-final", 
    format="zip", 
    root_dir="/kaggle/working/nllb-legal-final"
)

print("✅ Model zipped successfully! Check the 'Output' panel on the right side of Kaggle to download 'nllb-legal-final.zip'")
